# Review an AI-written ROAS analysis

*Data Analysis in Python · Lesson 14 final exercise*

A colleague asked an AI assistant for Q1 2025's ROAS by platform and conversion rate by channel group. The notebook below is what it wrote, and it runs without a single error. Its summary recommends moving budget to Meta. Before anyone acts on it, review it: three lines are wrong.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Fix the AI's cells where they are, and add your own cells above the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Run all, and read the AI's notebook from the top. Check each result the way topic 14.2. did: the rows, the totals and how each number is calculated.
2. Fix the cell that reads the exports, so the `sessions` table has every day of Q1 2025.
3. Fix the cell that creates the `channels` table, so its conversion rates are each channel group's transactions divided by its sessions.
4. Fix the cell that creates the `platforms` table, so each date and campaign's revenue is counted once. Make the merge stop if a date and campaign appear more than once on either side.
5. Add a cell with topic 14.3.'s `validate_report(sessions, report, start, end)`, and `assert` that every check passes for the `sessions` and `channels` tables.
6. Rewrite the summary cell with the right numbers.
7. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `sessions` table has 5,297 rows, every day of Q1 2025, with 324,563 sessions and \$655,182.61 of revenue. The `channels` table has 9 channel groups, with Email converting best at 6.22%. The `platforms` table's ROAS is Google Ads 2.47, Meta 2.33 and YouTube 1.89. Your `validate_report()` passes on your tables and fails on the AI's.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-01", "2025-02", "2025-03"]
START, END = "2025-01-01", "2025-03-31"  # Q1 2025

## The AI's notebook

Your colleague's prompt to the assistant was: *"Here are our GA4 session exports and our ad platforms' spend files. Write a notebook that reports Q1 2025's conversion rate by channel group and ROAS by platform, and summarize where we should move budget."* Every cell from here to the summary is what the assistant wrote, apart from the channel groups cell, which is the team's own.

In [ ]:
# Load the Q1 2025 session exports and clean them
frames = []
for month in MONTHS:
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    # Parse dates flexibly, so every export's date format works
    export["date"] = pd.to_datetime(export["date"], format="mixed")
    # Some exports store revenue as currency text, such as "$1,234.50"
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    frames.append(export)

raw = pd.concat(frames, ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()  # The monthly exports overlap by two days
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions.shape

In [ ]:
# The team's channel groups, from lesson 4. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

In [ ]:
# Conversion rate by channel group: average the daily rates, so one unusual day can't skew the result
daily = sessions.groupby(["channel_group", "date"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
daily["conversion_rate"] = daily["transactions"] / daily["sessions"]
channels = daily.groupby("channel_group", as_index=False).agg(
    sessions=("sessions", "sum"),
    transactions=("transactions", "sum"),
    revenue=("revenue", "sum"),
    conversion_rate=("conversion_rate", "mean"),
)
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
channels

In [ ]:
# One spend table from the three platforms' exports
google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])

google = google.rename(columns={"campaign_name": "campaign"})
google["spend"] = google["cost_micros"] / 1_000_000  # Google Ads reports cost in millionths of a dollar
google["platform"] = "Google Ads"
meta = meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Amount spent (USD)": "spend"})
meta["platform"] = "Meta"
youtube = youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Cost": "spend"})
youtube["platform"] = "YouTube"

columns = ["date", "platform", "campaign", "spend"]
spend = pd.concat([google[columns], meta[columns], youtube[columns]], ignore_index=True)
spend = spend[spend["date"].between(START, END)].reset_index(drop=True)
spend["campaign"] = spend["campaign"].str.strip().str.lower()  # Match the campaign names in GA4
spend.groupby("platform")["spend"].sum()

In [ ]:
# Revenue per date and campaign from GA4, joined to the spend for ROAS
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = revenue["campaign"].str.strip().str.lower()

campaign_days = spend.merge(revenue, on=["date", "campaign"], how="left")
platforms = campaign_days.groupby("platform")[["spend", "revenue"]].sum()
platforms["roas"] = platforms["revenue"] / platforms["spend"]
platforms.round(2)

**Summary.** In Q1 2025, Meta returned \$5.83 for every dollar it spent, more than twice Google Ads' \$2.37 and three times YouTube's \$1.66. Meta is clearly the most efficient platform, and I'd move budget to it from YouTube first, then from Google Ads. Among channel groups, Email converts best, at 6.54% of sessions.

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    message = f"{type(error).__name__}: {error}" if error is not None else ""
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif "AssertionError" in message:
        results.append((None, f"Your assert stopped the notebook: {error}. A check failed, so a table is still wrong. Fix it, then choose Runtime → Restart session and run all."))
    elif "one-to-one" in message:
        results.append((None, "The merge stopped: a date and campaign appear more than once. That's what validate=\"one_to_one\" is for. Total the spend by date, platform and campaign before you merge."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    sessions = found.get("sessions")
    if not isinstance(sessions, pd.DataFrame) or [column for column in ["date", "sessions", "revenue", "channel_group"] if column not in sessions.columns]:
        results.append((False, "sessions should be the clean sessions table, with a channel_group column."))
    else:
        days, total = sessions["date"].nunique(), int(sessions["sessions"].sum())
        if days == 81 or total == 305686:
            results.append((False, f"The sessions table has {days} of Q1's 90 days and {total:,} sessions. format=\"mixed\" read March's day-first dates month first: 05/03/2025 became May 3. Read March 2025 with format=\"%d/%m/%Y\"."))
        elif days != 90 or total != 324563 or abs(float(sessions["revenue"].sum()) - 655182.61) > 0.05:
            results.append((False, f"Q1 2025 has 90 days, 324,563 sessions and $655,182.61 of revenue. The sessions table has {days} days, {total:,} sessions and ${float(sessions['revenue'].sum()):,.2f}. Check how each month's dates are read."))
        else:
            results.append((True, "The sessions table has every day of Q1 2025: 324,563 sessions and $655,182.61 of revenue."))

    channels = found.get("channels")
    columns = ["channel_group", "sessions", "transactions", "revenue", "conversion_rate"]
    if not isinstance(channels, pd.DataFrame) or [column for column in columns if column not in channels.columns]:
        results.append((False, f"channels should be a DataFrame with the columns {', '.join(columns)}."))
    elif len(channels) != 9 or int(channels["sessions"].sum()) != 324563:
        results.append((False, f"The channels table should have 9 channel groups and 324,563 sessions. It has {len(channels)} and {int(channels['sessions'].sum()):,}. Fix the sessions table first."))
    else:
        email = channels.loc[channels["channel_group"] == "Email", "conversion_rate"]
        pooled = channels["transactions"] / channels["sessions"]
        if ((channels["conversion_rate"] - pooled).abs() > 0.00001).any():
            rate = float(email.iloc[0]) if len(email) else float("nan")
            results.append((False, f"The conversion rates aren't transactions divided by sessions: Email's is {rate:.2%}, and its totals give {1659 / 26659:.2%}. That's a mean of daily rates. Divide the totals after grouping."))
        else:
            results.append((True, "9 channel groups, with conversion rates from totals: Email converts best, at 6.22%."))

    platforms = found.get("platforms")
    expected = {"Google Ads": 2.4695, "Meta": 2.3315, "YouTube": 1.8937}
    if not isinstance(platforms, pd.DataFrame) or "roas" not in platforms.columns:
        results.append((False, "platforms should be a DataFrame with a roas column, and one row per platform."))
    else:
        table = platforms.set_index("platform") if "platform" in platforms.columns else platforms
        roas = {name: float(table.loc[name, "roas"]) for name in expected if name in table.index}
        merged_with_check = isinstance(inputs, list) and any("one_to_one" in source for source in inputs[1:-1])
        if len(roas) != 3:
            results.append((False, "The platforms table should have one row per platform: Google Ads, Meta and YouTube."))
        elif roas["Meta"] > 4:
            results.append((False, f"Meta's ROAS is {roas['Meta']:.2f}. Meta's export has a row per ad set, so the merge copied each campaign-day's revenue to every ad set. Total the spend by date, platform and campaign before you merge."))
        elif [name for name in expected if abs(roas[name] - expected[name]) > 0.0005]:
            results.append((False, "The ROAS should be Google Ads 2.47, Meta 2.33 and YouTube 1.89. Fix the sessions table first: the revenue comes from it."))
        elif not merged_with_check:
            results.append((False, "The ROAS is right. Make the merge stop if a date and campaign appear more than once: validate=\"one_to_one\"."))
        else:
            results.append((True, "ROAS by platform: Google Ads 2.47, Meta 2.33 and YouTube 1.89. Meta doesn't beat Google Ads."))

    validate_report = found.get("validate_report")
    if not callable(validate_report):
        results.append((False, "Add topic 14.3.'s validate_report(sessions, report, start, end), and assert its checks."))
    elif isinstance(sessions, pd.DataFrame) and isinstance(channels, pd.DataFrame):
        good = validate_report(sessions, channels, "2025-01-01", "2025-03-31")
        missing_days = sessions[~(sessions["date"].dt.month.eq(3) & sessions["date"].dt.day.between(4, 12))]
        mean_rates = channels.copy()
        mean_rates["conversion_rate"] = mean_rates["conversion_rate"] * 1.02
        caught_days = not all(validate_report(missing_days, channels, "2025-01-01", "2025-03-31").values())
        caught_rates = not all(validate_report(sessions, mean_rates, "2025-01-01", "2025-03-31").values())
        asserted = isinstance(inputs, list) and any("assert" in source and "validate_report(" in source for source in inputs[1:-1])
        if not isinstance(good, dict) or not all(good.values()):
            results.append((False, f"validate_report() should pass on your sessions and channels tables. It returns {good}."))
        elif not caught_days:
            results.append((False, "validate_report() passes a sessions table that's missing March 4 to 12. Check that every day from start to end is there."))
        elif not caught_rates:
            results.append((False, "validate_report() passes conversion rates that aren't transactions divided by sessions. Check the rates against the totals."))
        elif not asserted:
            results.append((False, "validate_report() works. Make a failure stop the notebook: assert all(checks.values()), checks."))
        else:
            results.append((True, "validate_report() passes on your tables, and fails on the AI's missing days and mean rates."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: where to look</b></summary>

Topic 14.2.'s three questions, for every result. Did rows appear or disappear? Count the days in the `sessions` table: Q1 2025 has 90. Do the totals match their source? Compare the revenue in the `campaign_days` table with the revenue of the same days and campaigns in the `revenue` table. Is each number calculated the way it's defined? Divide the `channels` table's transactions by its sessions, and compare.

~~~python
sessions["date"].nunique(), len(pd.date_range(START, END))
len(spend), len(campaign_days)
channels["transactions"] / channels["sessions"] - channels["conversion_rate"]
~~~

</details>

<details>
<summary><b>Step 2: every day of the quarter</b></summary>

Topic 3.2.: March 2025's export writes its dates day first, such as `13/03/2025`. `format="mixed"` guesses each date on its own, so `05/03/2025` becomes May 3. Give each month its own format.

~~~python
DAY_FIRST_MONTHS = ["2025-03"]
...
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else ...
    export["date"] = pd.to_datetime(export["date"], format=date_format)
~~~

</details>

<details>
<summary><b>Step 3: rates from totals</b></summary>

Topic 4.4.: a channel group's conversion rate is its transactions divided by its sessions, after grouping. A mean of daily rates gives a quiet day as much say as a busy one.

~~~python
channels = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
channels["conversion_rate"] = ... / ...
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
~~~

</details>

<details>
<summary><b>Step 4: one row per campaign-day</b></summary>

Topic 5.4.: Meta has a row per ad set, so total the spend by date, platform and campaign before you join, and make the merge one to one.

~~~python
daily_spend = spend.groupby([..., ..., ...], as_index=False)["spend"].sum()
campaign_days = daily_spend.merge(revenue, on=["date", "campaign"], how="left", validate=...)
~~~

</details>

<details>
<summary><b>Step 5: the validation cell</b></summary>

Topic 14.3.'s function, unchanged, and an `assert` that prints the checks when one fails.

~~~python
def validate_report(sessions, report, start, end):
    """Return each check's name and whether the report passes it."""
    return {
        "every day is there": ...,
        "sessions add up": ...,
        "revenue adds up": ...,
        "rates from totals": ...,
    }


checks = validate_report(sessions, channels, START, END)
assert all(checks.values()), checks
~~~

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
# The AI's first cell, fixed: March 2025's export is day first
DAY_FIRST_MONTHS = ["2025-03"]
frames = []
for month in MONTHS:
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    frames.append(export)

raw = pd.concat(frames, ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions.shape, sessions["date"].nunique()
```

```python
# The channels cell, fixed: rates from totals
channels = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
channels["conversion_rate"] = channels["transactions"] / channels["sessions"]
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
channels
```

```python
# The platforms cell, fixed: one row per date, platform and campaign before the merge
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = revenue["campaign"].str.strip().str.lower()

daily_spend = spend.groupby(["date", "platform", "campaign"], as_index=False)["spend"].sum()
campaign_days = daily_spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
platforms = campaign_days.groupby("platform")[["spend", "revenue"]].sum()
platforms["roas"] = platforms["revenue"] / platforms["spend"]
platforms.round(2)
```

```python
def validate_report(sessions, report, start, end):
    """Return each check's name and whether the report passes it."""
    return {
        "every day is there": sessions["date"].nunique() == len(pd.date_range(start, end)),
        "sessions add up": bool(report["sessions"].sum() == sessions["sessions"].sum()),
        "revenue adds up": bool(round(report["revenue"].sum(), 2) == round(sessions["revenue"].sum(), 2)),
        "rates from totals": bool(((report["conversion_rate"] - report["transactions"] / report["sessions"]).abs() < 0.000001).all()),
    }


checks = validate_report(sessions, channels, START, END)
assert all(checks.values()), checks
checks
```

```markdown
**Summary.** In Q1 2025, Google Ads returned \$2.47 for every dollar it spent, Meta \$2.33 and YouTube \$1.89. Meta doesn't beat Google Ads, so the case for moving budget to it is gone. YouTube returns the least, and that's where I'd look first. Among channel groups, Email converts best, at 6.22% of sessions.
```

</details>